In [40]:
import os
import numpy as np

from PIL import Image
%load_ext autoreload
%autoreload 2

# Make a dataloader that mixes wevrything randomly loads the images splits a train and test and computes the embeddings
from pathlib import Path

from data_handler import EleHandler
from seek_code import SEEK

import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler
import timm


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
def closest_valid_one_hot(prob_vector):
    """
    Converts a probability tensor into the closest valid one-hot encoded representation for each attribute.
    """
    closest_one_hot = []
    index = 0

    for group in SEEK.attribute_names:
        slice_length = SEEK.lengths[group]
        prob_slice = prob_vector[:, index:index + slice_length]
        one_hot_slice = torch.zeros_like(prob_slice)
        
        max_indices = torch.argmax(prob_slice, dim=1)
        one_hot_slice[torch.arange(prob_slice.size(0)), max_indices] = 1.0
        closest_one_hot.append(one_hot_slice)
        
        index += slice_length

    return torch.cat(closest_one_hot, dim=1)

In [93]:
def calculate_attribute_accuracy(predicted, labels):
    # Convert predicted logits to closest valid one-hot representation
    predicted_one_hot = closest_valid_one_hot(predicted)
    
    # Initialize accuracy tracking for all attributes, including individual and averaged
    accuracies = {name: 0 for name in SEEK.attribute_names}

    index = 0
    for name in SEEK.attribute_names:
        length = SEEK.lengths[name]
        pred_slice = predicted_one_hot[:, index:index + length]
        label_slice = labels[:, index:index + length]

        # Calculate per-sample accuracy
        pred_indices = torch.argmax(pred_slice, dim=1)
        label_indices = torch.argmax(label_slice, dim=1)
        accuracies[name] = (pred_indices == label_indices).float().mean().item()

        index += length

    # Include accuracy for whole SEEK code prediction
    correct_whole_code = torch.all(predicted_one_hot == labels, dim=1).float().mean().item()
    accuracies['average'] = sum(accuracies.values()) / len(accuracies)
    accuracies['whole_code'] = correct_whole_code

    return accuracies


In [94]:
def epoch_pass(epoch, loader, training=True):
    model.eval()  # Freeze model for inference
    layer.train() if training else layer.eval()

    total_loss = 0
    num_batches = len(loader)
    epoch_accuracies = {name: 0 for name in SEEK.attribute_names}
    epoch_accuracies['average'] = 0
    epoch_accuracies['whole_code'] = 0

    for batch in loader:
        images, subject_id, ele_id, identified, subject_SEEK, ele_SEEK = batch

        # Prepare labels and transfer them to GPU
        labels = torch.stack([SEEK(s).one_hot_encode() for s in subject_SEEK]).to('cuda')
        images = images.to('cuda')

        # Obtain embeddings from the frozen model
        with torch.no_grad():
            embeddings = model(images)

        # Forward pass through the trainable layer
        outputs = layer(embeddings)
        
        # Calculate loss
        loss = F.mse_loss(outputs, labels)
        total_loss += loss.item()

        # Calculate accuracy for average and whole-code
        predicted_SEEK = closest_valid_one_hot(outputs)
        batch_accuracies = calculate_attribute_accuracy(predicted_SEEK, labels)
        
        for name, value in batch_accuracies.items():            
            epoch_accuracies[name] += value

        # Backpropagation only for the linear layer if training
        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Calculate epoch-level average loss and accuracies
    average_loss = total_loss / num_batches
    epoch_accuracies = {name: accuracy / num_batches for name, accuracy in epoch_accuracies.items()}
    
    return average_loss, epoch_accuracies


In [ ]:
def train_concept_head(train_loader, val_loader, backbone_frozen=True, num_epochs=10):
    """
    Trains a linear layer on top of a pre-trained model to predict SEEK codes.
    """
    for epoch in range(num_epochs):
        train_loss, train_acc = epoch_pass(epoch, train_loader, training=True)
        val_loss, val_acc = epoch_pass(epoch, val_loader, training=False)
        
        if epoch % 5 == 0:
            print(f" Epoch {epoch + 1}/{num_epochs} - Loss : train {train_loss:.4f} val {val_loss:.4f} - Avg accuracy : train {train_acc['average'] * 100:.2f}% val {val_acc['average'] * 100:.2f}% - Whole Code : train {train_acc['whole_code'] * 100:.2f}% val {val_acc['whole_code'] * 100:.2f}%")
    return train_loss, train_acc['average'], train_acc['whole_code'], val_loss, val_acc['average'], val_acc['whole_code']

def test_concept_head(test_loader):

    test_loss, test_acc = epoch_pass(-1, test_loader, training=False)
    
    print("============ Test Summary =========== ")
    print(f"  Test Loss: {test_loss:.4f}")

    # Detailed accuracy for each characteristic
    print("\nDetailed Characteristic Accuracies:")
    for name, value in test_acc.items():
        print(f"  {name} Accuracy: {value * 100:.2f}%")
    
    return test_loss, test_acc['average'], test_acc['whole_code'], test_acc


In [ ]:

# Create dataset and dataloaders
dataset = EleHandler(dataset_dir=Path('/archive/vision/beery/animal_reid/datasets/elephants_zooniverse'),
                       dictonary_path=Path('/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary.csv'),
                       transform='MegaDescriptor-elephant'
                       )


train_size = int(0.7 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size  # Ensure the sum equals the length of the dataset

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=1024, sampler=RandomSampler(train_dataset))
val_loader = DataLoader(val_dataset, batch_size=1024, sampler=RandomSampler(val_dataset))
test_loader = DataLoader(test_dataset, batch_size=1024, sampler=RandomSampler(test_dataset))


model = timm.create_model("hf-hub:BVRA/MegaDescriptor-T-224", pretrained=True, num_classes=0) # Mega-Descriptor pretrained
#state_dict = torch.load("../weights/forest_elephants-reid_weights.pt", map_location='cuda') # Pretrained on forest elephants
state_dict = torch.load("../weights/savanna_elephants_md_v2_epoch_60.pt", map_location='cuda') # Pretrained on savannah elephants
model.load_state_dict(state_dict["model"]) if "optimizer" in state_dict else model.load_state_dict(state_dict)
model.to('cuda')

layer = torch.nn.Sequential(
    torch.nn.Linear(768, 63),
    torch.nn.Sigmoid()
).to('cuda')

optimizer = torch.optim.Adam(layer.parameters(), lr=0.001)

train_concept_head(train_loader, val_loader, backbone_frozen = True,  num_epochs=50)
test_concept_head(test_loader)

/tmp/ipykernel_3782822/569609807.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("../weights/forest_elephants-reid_weights.pt", map_location='cu

 Epoch 1/50 - TrainLoss: 0.1504 - train_acc: 61.19% - train_acc_perfect: 0.67%
  Validation Loss: 0.1166 -  Average Concept Accuracy: 67.19% - Whole-Code Accuracy: 2.21%
 Epoch 2/50 - TrainLoss: 0.1111 - train_acc: 68.39% - train_acc_perfect: 2.00%
  Validation Loss: 0.1109 -  Average Concept Accuracy: 68.69% - Whole-Code Accuracy: 1.77%
 Epoch 3/50 - TrainLoss: 0.1067 - train_acc: 69.96% - train_acc_perfect: 1.87%
  Validation Loss: 0.1081 -  Average Concept Accuracy: 69.58% - Whole-Code Accuracy: 1.81%
 Epoch 4/50 - TrainLoss: 0.1042 - train_acc: 70.86% - train_acc_perfect: 2.85%
  Validation Loss: 0.1070 -  Average Concept Accuracy: 70.13% - Whole-Code Accuracy: 2.75%
 Epoch 5/50 - TrainLoss: 0.1025 - train_acc: 71.49% - train_acc_perfect: 3.17%
  Validation Loss: 0.1052 -  Average Concept Accuracy: 70.69% - Whole-Code Accuracy: 2.94%
 Epoch 6/50 - TrainLoss: 0.1010 - train_acc: 71.94% - train_acc_perfect: 3.48%
  Validation Loss: 0.1044 -  Average Concept Accuracy: 70.88% - Whole-C

KeyboardInterrupt: 